# Noise-corrected backboning of the conversation graphs

```
conversation graph edges (.pkl, notebook 03)
    ↓  Stage 1 · noise-corrected (NC) score per edge, per instance
NC table   [instance, source, target, nij, score, sdev_cij]
    ↓  Stage 2 · per-instance δ at the elbow of the edge-retention curve, then thresholding
backbone edges (.pkl)   → --edges input of train_DeTox-Fed.py
```

Stage 2b is an alternative rule for choosing δ (minimum edge retention) and is not
needed for training. The backboning functions come from `backboning_third_party_code/backboning.py`
(Coscia & Neffke, *Network Backboning with Noisy Data*, ICDE 2017).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binom

sys.path.insert(0, str(Path("backboning_third_party_code").resolve()))
import backboning

## Configuration

In [ ]:
DATASET = "pleroma"  # or "mastodon"
DATA_DIR = Path("../../data") / DATASET
OUTPUT_DIR = DATA_DIR / "backboning"
FIGURES_DIR = Path("../../figures") / DATASET / "backboning"

INPUT_EDGES_PKL = DATA_DIR / f"{DATASET}_intra_edges_1_common_user.pkl"
BACKBONE_EDGES_OUT_PKL = DATA_DIR / f"{DATASET}_intra_edges_1_common_user_nc_backbone.pkl"

# Stage 1 outputs
NC_EDGES_OUT_PKL = OUTPUT_DIR / f"{DATASET}_intra_edges_1_common_user_noise_corrected.pkl"
NC_STATS_CSV_OUT = OUTPUT_DIR / f"{DATASET}_noise_corrected_stats_per_instance.csv"

# Stage 2 outputs
DELTA_CSV_OUT = OUTPUT_DIR / f"{DATASET}_delta_elbow_per_instance.csv"
DENSITIES_OUT_PKL = OUTPUT_DIR / f"{DATASET}_delta_sweep_curves.pkl"

# Stage 2b outputs
MIN_EDGES_PCT = 80.0
BACKBONE_RET_OUT_PKL = OUTPUT_DIR / f"{DATASET}_nc_backbone_retention{int(MIN_EDGES_PCT)}.pkl"
DELTA_RET_CSV_OUT = OUTPUT_DIR / f"{DATASET}_delta_retention{int(MIN_EDGES_PCT)}_per_instance.csv"
DENSITIES_RET_OUT_PKL = OUTPUT_DIR / f"{DATASET}_delta_sweep_curves_retention{int(MIN_EDGES_PCT)}.pkl"

# δ sweep
SWEEP_START = 1.0
SWEEP_END = 8.0
SWEEP_STEP = 0.05

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Stage 1 · Noise correction

`noise_corrected()` extends `backboning.noise_corrected` with string node IDs,
optional aggregation of duplicate undirected pairs, and `NaN` (instead of an error)
for the standard deviation of degenerate edges.

In [ ]:
def noise_corrected(
    table: pd.DataFrame,
    undirected: bool = False,
    return_self_loops: bool = False,
    calculate_p_value: bool = False,
    aggregate_undirected_duplicates: bool = True,
) -> pd.DataFrame:
    """
    Noise-corrected score and its standard deviation (``sdev_cij``) per edge.

    Required columns: ``src``, ``trg``, ``nij`` (numeric weight).
    With ``undirected=True`` each edge is stored as (min, max) and, if
    ``aggregate_undirected_duplicates``, duplicate pairs are summed.
    """
    table = table.copy()
    for col in ("src", "trg", "nij"):
        if col not in table.columns:
            raise ValueError(f"Expected column '{col}' not found.")
    table["src"] = table["src"].astype(str)
    table["trg"] = table["trg"].astype(str)
    table["nij"] = pd.to_numeric(table["nij"], errors="raise")

    if not return_self_loops:
        table = table[table["src"] != table["trg"]].copy()

    if undirected:
        a = table["src"].astype("string")
        b = table["trg"].astype("string")
        table["src"] = np.where(a <= b, a, b)
        table["trg"] = np.where(a <= b, b, a)
        if aggregate_undirected_duplicates:
            table = table.groupby(["src", "trg"], as_index=False, sort=False)["nij"].sum()

    src_sum = table.groupby("src")["nij"].sum().to_frame("ni.")
    trg_sum = table.groupby("trg")["nij"].sum().to_frame("n.j")
    table = table.merge(src_sum, left_on="src", right_index=True)
    table = table.merge(trg_sum, left_on="trg", right_index=True)

    table["n.."] = table["nij"].sum()
    n_tot = float(table["n.."].iloc[0]) if len(table) else 0.0
    if n_tot <= 0:
        return table[["src", "trg", "nij"]].assign(score=np.nan, sdev_cij=np.nan)

    table["mean_prior_probability"] = (table["ni."] * table["n.j"]) / (n_tot ** 2)

    if calculate_p_value:
        p = table["mean_prior_probability"].clip(0, 1)
        table["score"] = binom.cdf(table["nij"].astype(int), int(n_tot), p)
        return table[["src", "trg", "nij", "score"]]

    table["kappa"] = n_tot / (table["ni."] * table["n.j"])
    x = table["kappa"] * table["nij"]
    table["score"] = (x - 1) / (x + 1)

    table["var_prior_probability"] = (
        (table["ni."] * table["n.j"] * (n_tot - table["ni."]) * (n_tot - table["n.j"]))
        / ((n_tot ** 5) * (n_tot - 1))
    )

    for c in ["alpha_prior", "beta_prior", "alpha_post", "beta_post",
              "expected_pij", "variance_nij", "d", "variance_cij", "sdev_cij"]:
        table[c] = np.nan

    ok = (table["var_prior_probability"] > 0) & np.isfinite(table["var_prior_probability"])
    if ok.any():
        t = table.loc[ok]
        m = t["mean_prior_probability"]
        v = t["var_prior_probability"]
        alpha_prior = ((m ** 2 / v) * (1 - m)) - m
        beta_prior = ((m / v) * (1 - m ** 2)) - (1 - m)
        alpha_post = alpha_prior + t["nij"]
        beta_post = (n_tot - t["nij"]) + beta_prior
        expected_pij = alpha_post / (alpha_post + beta_post)
        variance_nij = expected_pij * (1 - expected_pij) * n_tot
        denom = t["ni."] * t["n.j"]
        d = (1.0 / denom) - (n_tot * ((t["ni."] + t["n.j"]) / (denom ** 2)))
        variance_cij = variance_nij * (
            ((2 * (t["kappa"] + (t["nij"] * d))) / (((t["kappa"] * t["nij"]) + 1) ** 2)) ** 2
        )
        table.loc[ok, "sdev_cij"] = np.sqrt(variance_cij)

    return table[["src", "trg", "nij", "score", "sdev_cij"]]

In [ ]:
df_raw = pd.read_pickle(INPUT_EDGES_PKL)
if "instance_id" in df_raw.columns and "instance_toot_found_id" not in df_raw.columns:
    df_raw = df_raw.rename(columns={"instance_id": "instance_toot_found_id"})

for col in ["source", "target", "weight", "instance_toot_found_id"]:
    if col not in df_raw.columns:
        raise ValueError(f"Expected column '{col}' not found.")

df_raw["source"] = df_raw["source"].astype(str)
df_raw["target"] = df_raw["target"].astype(str)
df_raw["weight"] = pd.to_numeric(df_raw["weight"], errors="raise")
print(f"{len(df_raw):,} edges across {df_raw['instance_toot_found_id'].nunique():,} instances")

nc_stats, nc_records = [], []
for inst, df_inst in df_raw.groupby("instance_toot_found_id", sort=False):
    edge_table = df_inst.rename(columns={"source": "src", "target": "trg", "weight": "nij"})[["src", "trg", "nij"]]
    nc = noise_corrected(edge_table, undirected=True, aggregate_undirected_duplicates=True)

    nc_stats.append({
        "instance_toot_found_id": inst,
        "orig_nodes": len(set(edge_table["src"]) | set(edge_table["trg"])),
        "orig_edges": len(edge_table),
        "nc_nodes": len(set(nc["src"]) | set(nc["trg"])) if len(nc) else 0,
        "nc_edges": len(nc),
        "nan_sdev_cij": int(nc["sdev_cij"].isna().sum()),
    })
    nc_records.append(nc.rename(columns={"src": "source", "trg": "target"}).assign(instance_toot_found_id=inst))

pd.DataFrame(nc_stats).to_csv(NC_STATS_CSV_OUT, index=False)
nc_all = (
    pd.concat(nc_records, ignore_index=True)
    if nc_records
    else pd.DataFrame(columns=["source", "target", "nij", "score", "sdev_cij", "instance_toot_found_id"])
)
nc_all.to_pickle(NC_EDGES_OUT_PKL)
print(f"Saved {len(nc_all):,} NC edges → {NC_EDGES_OUT_PKL}")

## Stage 2 · Backbone extraction

For each instance, δ is swept from `SWEEP_START` to `SWEEP_END`; the chosen δ is the
elbow of the edges-retained curve (point furthest from the chord between its ends).

In [ ]:
def graph_density_undirected(n_nodes, n_edges):
    if n_nodes is None or n_nodes < 2:
        return 0.0
    return (2.0 * n_edges) / (n_nodes * (n_nodes - 1))


def count_nodes_from_edges(df, src="src", trg="trg"):
    if df is None or len(df) == 0:
        return 0
    return int(pd.Index(df[src]).append(pd.Index(df[trg])).nunique())


def delta_sweep(nc_table, start=SWEEP_START, end=SWEEP_END, step=SWEEP_STEP):
    rows = list(backboning.test_densities(nc_table, start, end, step))
    return pd.DataFrame(rows, columns=["delta", "nodes", "nodes_pct", "edges", "edges_pct", "avgdeg", "rel_avgdeg"])


def choose_delta_elbow(nc_table):
    """Returns (delta, sweep table)."""
    dens = delta_sweep(nc_table)
    if dens.empty:
        return SWEEP_START, dens

    x = dens["delta"].to_numpy(dtype=float)
    y = dens["edges_pct"].to_numpy(dtype=float)
    if np.allclose(y, y[0]):
        return float(dens.loc[0, "delta"]), dens

    x_n = (x - x.min()) / (x.max() - x.min() + 1e-12)
    y_n = (y - y.min()) / (y.max() - y.min() + 1e-12)
    x1, y1, x2, y2 = x_n[0], y_n[0], x_n[-1], y_n[-1]
    dist = np.abs((y2 - y1) * x_n - (x2 - x1) * y_n + x2 * y1 - y2 * x1) / (np.hypot(x2 - x1, y2 - y1) + 1e-12)
    return float(dens.loc[int(dist.argmax()), "delta"]), dens


def choose_delta_for_edge_retention(nc_table, min_edges_pct=MIN_EDGES_PCT):
    """Largest δ that keeps at least ``min_edges_pct``% of edges (falls back to SWEEP_START)."""
    dens = delta_sweep(nc_table)
    if dens.empty:
        return SWEEP_START, dens
    ok = dens[dens["edges_pct"] >= min_edges_pct]
    return (float(ok["delta"].max()) if len(ok) else float(SWEEP_START)), dens


def extract_backbones(nc_all, choose_delta):
    """Apply ``choose_delta`` and thresholding per instance."""
    delta_rows, bb_records, curves = [], [], []

    for inst, g in nc_all.groupby("instance_toot_found_id", sort=False):
        # backboning.thresholding expects the columns src, trg, score and sdev_cij.
        nc = g.rename(columns={"source": "src", "target": "trg"}).copy()
        nc["src"] = nc["src"].astype(str)
        nc["trg"] = nc["trg"].astype(str)

        delta, dens = choose_delta(nc)
        bb = backboning.thresholding(nc, delta)

        nc_edges, nc_nodes = len(nc), count_nodes_from_edges(nc)
        bb_edges, bb_nodes = len(bb), count_nodes_from_edges(bb)

        delta_rows.append({
            "instance_toot_found_id": inst,
            "delta": delta,
            "nc_nodes": nc_nodes,
            "nc_edges": nc_edges,
            "nc_avgdeg": (2.0 * nc_edges / nc_nodes) if nc_nodes else 0.0,
            "nc_density": graph_density_undirected(nc_nodes, nc_edges),
            "bb_nodes": bb_nodes,
            "bb_edges": bb_edges,
            "bb_avgdeg": (2.0 * bb_edges / bb_nodes) if bb_nodes else 0.0,
            "bb_density": graph_density_undirected(bb_nodes, bb_edges),
            "edges_retained_pct": (100.0 * bb_edges / nc_edges) if nc_edges else 0.0,
            "nodes_retained_pct": (100.0 * bb_nodes / nc_nodes) if nc_nodes else 0.0,
            "nan_sdev_cij": int(nc["sdev_cij"].isna().sum()),
            "score_min": float(nc["score"].min()) if nc_edges else np.nan,
            "score_max": float(nc["score"].max()) if nc_edges else np.nan,
        })

        if bb_edges:
            bb_records.append(bb.rename(columns={"src": "source", "trg": "target"}).assign(instance_toot_found_id=inst))
        curves.append(dens.assign(instance_toot_found_id=inst))

    delta_df = pd.DataFrame(delta_rows)
    bb_all = (
        pd.concat(bb_records, ignore_index=True)
        if bb_records
        else pd.DataFrame(columns=["source", "target", "nij", "score", "sdev_cij", "instance_toot_found_id"])
    )
    curves_df = pd.concat(curves, ignore_index=True) if curves else pd.DataFrame()
    return delta_df, bb_all, curves_df


def summarize(delta_df):
    nc_total, bb_total = delta_df["nc_edges"].sum(), delta_df["bb_edges"].sum()
    print(f"Instances with backbone edges: {(delta_df['bb_edges'] > 0).sum()} / {len(delta_df)}")
    print(f"NC edges:       {nc_total:,}")
    print(f"Backbone edges: {bb_total:,}")
    print(f"Edges removed:  {nc_total - bb_total:,} ({100 * (1 - bb_total / nc_total):.1f}%)")

In [ ]:
nc_all = pd.read_pickle(NC_EDGES_OUT_PKL)

delta_df, bb_all, curves = extract_backbones(nc_all, choose_delta_elbow)
delta_df.to_csv(DELTA_CSV_OUT, index=False)
bb_all.to_pickle(BACKBONE_EDGES_OUT_PKL)
curves.to_pickle(DENSITIES_OUT_PKL)

print(f"Backbone → {BACKBONE_EDGES_OUT_PKL}")
summarize(delta_df)

### Stage 2b · Retention-based backbone (alternative)

Per instance, the largest δ that still keeps at least `MIN_EDGES_PCT`% of the edges.

In [ ]:
delta_ret_df, bb_ret_all, curves_ret = extract_backbones(nc_all, choose_delta_for_edge_retention)
delta_ret_df.to_csv(DELTA_RET_CSV_OUT, index=False)
bb_ret_all.to_pickle(BACKBONE_RET_OUT_PKL)
curves_ret.to_pickle(DENSITIES_RET_OUT_PKL)
summarize(delta_ret_df)

## Descriptive analysis

These cells read the saved files and can be re-run on their own after the
configuration cell.

### Raw edge weights

In [ ]:
df_raw = pd.read_pickle(INPUT_EDGES_PKL)

weight_counts = df_raw["weight"].value_counts().sort_index()
weights_sorted = np.sort(df_raw["weight"].values)
ccdf_vals = 1.0 - np.arange(1, len(weights_sorted) + 1) / len(weights_sorted)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(weight_counts.index, weight_counts.values / weight_counts.sum())
axes[0].set_xlabel("Weight")
axes[0].set_ylabel("P(weight = w)")
axes[0].set_title("PMF of edge weights")

axes[1].step(weights_sorted, ccdf_vals, where="post")
axes[1].set_xlabel("Weight")
axes[1].set_ylabel("P(weight ≥ w)")
axes[1].set_title("CCDF of edge weights")
axes[1].set_ylim(0, 1)

axes[2].step(weights_sorted, ccdf_vals, where="post")
axes[2].set_xscale("log")
axes[2].set_yscale("log")
axes[2].set_xlabel("Weight (log)")
axes[2].set_ylabel("P(weight ≥ w) (log)")
axes[2].set_title("CCDF of edge weights (log-log)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "edge_weight_distributions.pdf", bbox_inches="tight")
plt.show()

prevalence = df_raw.groupby("weight")["instance_toot_found_id"].nunique().sort_index()
plt.figure(figsize=(8, 4))
plt.bar(prevalence.index, prevalence.values)
plt.xlabel("Weight")
plt.ylabel("Number of instances")
plt.title("Weight prevalence across instances")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "weight_prevalence_across_instances.pdf", bbox_inches="tight")
plt.show()

(
    df_raw.groupby("weight")
    .agg(total_edges=("weight", "count"), instances=("instance_toot_found_id", "nunique"))
    .sort_values("total_edges", ascending=False)
    .head(10)
)

In [ ]:
def cdf(arr):
    s = np.sort(arr)
    return s, np.arange(1, len(s) + 1) / len(s)


def ccdf(arr):
    s, p = cdf(arr)
    return s, 1 - p


PALETTE = {"before": "#2176AE", "after": "#F4A261"}
LW = 2.5
delta_df = pd.read_csv(DELTA_CSV_OUT)

### Graph density before and after backboning

In [ ]:
before = delta_df["nc_density"].to_numpy()
after = delta_df["bb_density"].to_numpy()
before, after = before[np.isfinite(before)], after[np.isfinite(after)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for label, arr, col in [("Before (NC)", before, PALETTE["before"]), ("After (backbone)", after, PALETTE["after"])]:
    x, y = cdf(arr)
    med = np.median(arr)
    ax1.step(x, y, where="post", color=col, linewidth=LW, label=label)
    ax1.axvline(med, color=col, linewidth=1.2, linestyle="--", alpha=0.7)
    ax1.scatter([med], [y[np.searchsorted(x, med, side="right") - 1]], color=col, s=65, zorder=5)

    x2, y2 = ccdf(arr)
    ax2.step(x2[y2 > 0], y2[y2 > 0], where="post", color=col, linewidth=LW, label=label)
    ax2.axvline(med, color=col, linewidth=1.2, linestyle="--", alpha=0.7)

ax1.set_xlabel("Graph density", fontsize=14)
ax1.set_ylabel("P(density < x)", fontsize=14)
ax1.set_title("CDF of graph density per instance")
ax1.set_ylim(0, 1.02)
ax2.set_xlabel("Graph density", fontsize=14)
ax2.set_ylabel("P(density > x)", fontsize=14)
ax2.set_title("CCDF of graph density per instance")
ax2.set_yscale("log")
for ax in (ax1, ax2):
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "density_cdf_ccdf.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for label, arr, col in [("Original", before, PALETTE["before"]), ("Noise-corrected backbone", after, PALETTE["after"])]:
    x, y = cdf(arr)
    med = np.median(arr)
    ax.step(x, y, where="post", color=col, linewidth=LW, label=label)
    ax.axvline(med, color=col, linewidth=1.2, linestyle="--", alpha=0.7)
    ax.scatter([med], [y[np.searchsorted(x, med, side="right") - 1]], color=col, s=65, zorder=5)

ax.set_xlabel(f"{DATASET.capitalize()} graph density", fontsize=20)
ax.set_ylabel("P(density < x)", fontsize=20)
ax.legend(fontsize=20)
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "density_cdf.pdf", dpi=300, bbox_inches="tight")
plt.show()

### Node and edge retention

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for label, column, col in [
    ("Nodes retained", "nodes_retained_pct", PALETTE["before"]),
    ("Edges retained", "edges_retained_pct", PALETTE["after"]),
]:
    arr = delta_df[column].to_numpy()
    arr = arr[np.isfinite(arr)]
    x, y = cdf(arr)
    med = np.median(arr)
    ax.step(x, y, where="post", linewidth=LW, color=col, label=label)
    ax.axvline(med, color=col, linewidth=1.2, linestyle="--", alpha=0.7)
    ax.scatter([med], [y[np.searchsorted(x, med, side="right") - 1]], color=col, s=65, zorder=5)

ax.set_xlabel(f"{DATASET.capitalize()} retention (%)", fontsize=20)
ax.set_ylabel("P(retention < x)", fontsize=20)
ax.legend(fontsize=20)
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "retention_cdf.pdf", dpi=300, bbox_inches="tight")
plt.show()

### Instances losing more than 5% of their nodes

In [ ]:
lost = (
    delta_df.loc[delta_df["nodes_retained_pct"] < 95, ["instance_toot_found_id", "nc_nodes", "bb_nodes", "nodes_retained_pct"]]
    .rename(columns={"instance_toot_found_id": "instance", "nc_nodes": "nodes_before", "bb_nodes": "nodes_after"})
    .sort_values("nodes_retained_pct")
    .reset_index(drop=True)
)
lost.to_csv(OUTPUT_DIR / "node_loss_instances.csv", index=False)
print(f"Instances with node loss > 5%: {len(lost)}")
lost